In [1]:
import json

with open('balakanda_chapter_1_triplets.json', 'r') as file:
    data = json.load(file)

parsed_data = []
for key, value in data[0].items():
    print(f"{value}")

```
('valmiki', 'asks_about', 'perfect_man')
('valmiki', 'interacts_with', 'narada')
('narada', 'is_a', 'celestial_sage')
('valmiki', 'is_a', 'ascetic')
('valmiki', 'is_in', 'canto_1')
('narada', 'is_in', 'canto_1')

```



In [2]:
def trippie(value):
    lines = value.splitlines()
    ans = []
    for line in lines:
        try:
            triplet = tuple(map(str.strip, line.split(')')[0].split('(')[-1].replace("'", "").replace('"', "").split(',')))
        except:
            print(line)
            continue
        ans.append(triplet)
    real_ans = []
    for trip in ans:
        #check if its actual triplet
        if len(trip) != 3:
            print("Not a triplet", trip)
            continue
        real_ans.append(trip)
    return real_ans

In [3]:
parsed_data = []
for ick in data:
    for key, value in ick.items():
        parsed_data.append({key: trippie(value)})
    

Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet ('',)
Not a triplet ('```',)
Not a triplet ('```',)
Not a triplet (''

In [4]:
len(parsed_data)

21

In [5]:
with open('parsed_balakanda_chapter_1_triplets.json', 'w') as file:
    json.dump(parsed_data, file, indent=4)

In [ ]:
for i in parsed_data:
    for key, value in i.items():
        print(key, len(value))

In [ ]:
from llama_index.core import KnowledgeGraphIndex
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.graph_stores.neo4j import Neo4jGraphStore
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from custom_llm import OurLLM
llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=None, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

In [ ]:
import json
with open('parsed_ayodhyakanda_triplets.json', 'r') as file:
    parsed_data = json.load(file)

file = 'ayodhyakanda.json'
import json
with open(file) as f:
    data = json.load(f)

# from llama_index.core import Document
# documents = []
# for i in data[:]:       
#     document = Document(
#     text = i['Prompt_Answer'],
#     metadata={
#         "canto": i['Canto'],
#         "canto_summary": i['Summary'],
#         "r_id": str(i['Canto']) + "_" + str(i['Prompt_Number'])
#     },
#     excluded_llm_metadata_keys=["r_id"],
#     metadata_seperator="::",
#     metadata_template="{key}=>{value}",
#     text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
#     )
#     documents.append(document)
# print(len(documents), len(parsed_data))
# print(f"Document Metadata: {documents[-1].metadata}")

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['Prompt_Answer'],
    metadata={
        "canto": i['Canto'],
        "canto_summary": i['Summary'],
        "r_id": str(i['Canto']) + "_" + str(i['Prompt_Number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{key}=>{value}",
    text_template="Content: {content}, Extra Chapter context for reference: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents), len(parsed_data))
print(f"Document Metadata: {documents[-1].metadata}")

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
node_parser = SentenceSplitter()
nodes = node_parser.get_nodes_from_documents(documents[:3])

In [ ]:
graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

index = KnowledgeGraphIndex([], 
                            storage_context=storage_context, 
                            include_embeddings=True)
from tqdm import tqdm
for i in tqdm(range(0, len(nodes))):
    if i > 0 and i % 30 == 0:
        index.storage_context.persist(persist_dir='not_this_time')
    for key,value in parsed_data[i].items():
        for tup in value:
            index.upsert_triplet_and_node(tuple(tup), nodes[i], include_embeddings=True)

In [ ]:
query_engine = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)

In [ ]:
response = query_engine.query("What did u understand from this?")

In [ ]:
print(response)